In [1]:
import os
import numpy as np
import tensorflow as tf
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

TFM_DIR = os.path.expanduser("~/Desktop/TFM")

# --- Redes: cargar el modelo guardado y predecir sobre prueba ---
datos = np.load(os.path.join(TFM_DIR, "datos_procesados_completo_2canales.npz"))
X_test, y_test = datos["X_test_norm"], datos["y_test"]

for nombre, archivo, salida in [
    ("CNN-1D", "modelo_final_cnn1d_completo_v1.keras", "roc_cnn1d.npz"),
    ("EEGNet", "modelo_final_eegnet_completo_v2_final.keras", "roc_eegnet.npz"),
]:
    modelo = tf.keras.models.load_model(os.path.join(TFM_DIR, archivo))
    prob = modelo.predict(X_test, batch_size=64).ravel()
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)
    np.savez(os.path.join(TFM_DIR, salida), fpr=fpr, tpr=tpr, auc=auc)
    print(f"{nombre}: AUC = {auc:.4f}")

# --- SVM lineal: reajustar con las características ya calculadas ---
f = np.load(os.path.join(TFM_DIR, "caracteristicas_svm_completo.npz"))
scaler = StandardScaler().fit(f["F_train"])
S_train, S_test = scaler.transform(f["F_train"]), scaler.transform(f["F_test"])

svm = LinearSVC(C=0.01, class_weight="balanced", dual=False, max_iter=5000, random_state=42)
svm.fit(S_train, f["y_train"])
scores = svm.decision_function(S_test)
fpr, tpr, _ = roc_curve(f["y_test"], scores)
auc = roc_auc_score(f["y_test"], scores)
np.savez(os.path.join(TFM_DIR, "roc_svm_lineal.npz"), fpr=fpr, tpr=tpr, auc=auc)
print(f"SVM lineal: AUC = {auc:.4f}")

699/699 ━━━━━━━━━━━━━━━━━━━━ 57s 81ms/step
CNN-1D: AUC = 0.9946
699/699 ━━━━━━━━━━━━━━━━━━━━ 86s 123ms/step
EEGNet: AUC = 0.9946
SVM lineal: AUC = 0.9751


In [4]:
import os
import numpy as np
from sklearn.metrics import roc_curve, roc_auc_score

scores_rbf = svm_rbf.decision_function(X_test)
fpr, tpr, _ = roc_curve(y_test, scores_rbf)
auc_rbf = roc_auc_score(y_test, scores_rbf)
print(f"SVM RBF: AUC = {auc_rbf:.4f}")   # debería salir ~0.988

np.savez(os.path.join(TFM_DIR, "roc_svm_rbf.npz"), fpr=fpr, tpr=tpr, auc=auc_rbf)

NameError: name 'svm_rbf' is not defined

In [3]:
import os
import numpy as np
from sklearn.metrics import roc_curve, roc_auc_score

# 'modelo_rbf' = el pipeline/clasificador ya ajustado (Nystroem + clasificador lineal)
# 'F_test_sc'  = características de prueba ya escaladas con el StandardScaler de entrenamiento
scores_rbf = svm_rbf.decision_function(F_test_sc)

fpr, tpr, _ = roc_curve(y_test, scores_rbf)
auc_rbf = roc_auc_score(y_test, scores_rbf)
print(f"SVM RBF: AUC = {auc_rbf:.4f}")   # debería salir ~0.988

np.savez(os.path.join(TFM_DIR, "roc_svm_rbf.npz"), fpr=fpr, tpr=tpr, auc=auc_rbf)

NameError: name 'svm_rbf' is not defined